# Head Motor Quickstart

Step-by-step bring-up of the two ST3215 head motors. Run cells top to bottom. Each section has a note telling you what 'success' looks like before moving on.

**Mechanism**: 40-tooth crown bevel + 20-tooth pinion → gear ratio 2.0. Motors same direction = tilt; opposite = pan.

**Hardware needed**:
- Waveshare USB-to-serial adapter plugged into the Jetson
- Two ST3215 servos wired to the bus
- External 7.4–12 V power supply for the servos (USB alone does NOT power them)

## 0. Setup

Change to project root so `python -m companion.motor.cli ...` resolves. Also make sure your venv is active in the terminal:

```
cd ~/Desktop/robotic-companion
source companion_env/bin/activate
```

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path('/home/yogee/Desktop/robotic-companion')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'cwd: {os.getcwd()}')

## 1. Sim sanity check (no hardware)

Confirm the code works end-to-end before touching hardware. Should print commanded tick values with no errors.

In [ ]:
!python -m companion.motor.cli test --sim --pan 25 --tilt -10

## 2. Detect the adapter

The USB adapter should show up as `/dev/ttyACM0` (Waveshare CH343 chip) or `/dev/ttyUSB0` (FTDI/CH340). If nothing appears, the adapter isn't plugged in or the driver didn't bind.

In [ ]:
!ls /dev/ttyUSB* /dev/ttyACM* 2>/dev/null
!ls /dev/serial/by-id/ 2>/dev/null

## 3. Scan the bus

With motors **powered on**, this should list both servo IDs.

**Expected outputs**:
- `Servos found on /dev/ttyACM0 @ 1000000 baud: [1, 2]` ✓ ideal
- `[1]` only — both servos ship at ID 1; run section 4 to reassign one
- `[]` with comm warnings — either motors aren't powered, or servos at unexpected baud. See troubleshooting section 10.

**Voltage check**: the readout shows `volt=…V`. Should be ~7.4–12 V. Below 7 V will cause alarms and flaky comms.

In [ ]:
!python -m companion.motor.cli scan

## 4. Assign IDs (only if scan showed `[1]`)

Both servos ship at ID 1, so they collide when both are on the bus. To separate them:

1. **Unplug one servo** from the bus (leave the other powered and connected)
2. Run the cell below — it reassigns the connected one from ID 1 → ID 2 and writes it to EEPROM so it persists across power cycles
3. **Reconnect the other servo**
4. Re-run the scan cell in section 3 — should show `[1, 2]`

Skip this section if scan already shows `[1, 2]`.

In [ ]:
!python -m companion.motor.cli assign-id --from 1 --to 2

## 5. Force single-turn mode

ST3215 servos can be shipped in wheel/multi-turn mode, where positions read outside the normal 0–4095 range and small moves become huge rotations. This command rewrites the EEPROM so both servos act as bounded single-turn (0–4095).

**After this cell**: power-cycle the servos (pull the power connector, wait 2 s, plug back in). EEPROM changes need a power-cycle to fully take effect.

In [ ]:
!python -m companion.motor.cli set-single-turn

## 5b. Reset safety limits

Restores sensible voltage / temperature / torque thresholds in EEPROM. Fixes servos that have bad values like `MAX_INPUT_VOLTAGE = 80` (8.0 V), which latches a persistent overvoltage alarm at normal 12 V supplies — the alarm in turn triggers weird fail-safe spinning when torque enables.

**After this cell: power-cycle the servos again.**

**Expected values after the cell + power-cycle** (visible in section 7):
```
MAX_INPUT_VOLTAGE = 140   (14.0 V)
MIN_INPUT_VOLTAGE = 40    (4.0 V)
MAX_TEMPERATURE   = 70    (°C)
MAX_TORQUE        = 1000
```

And the `0x01 servo error byte` warnings should stop appearing on subsequent operations.

In [ ]:
!python -m companion.motor.cli reset-safety-limits

## 6. Recenter the servos

Resets each servo's internal multi-turn counter. Wherever the head is *physically* when you press ENTER becomes the new tick 2048 (mechanical center).

**Procedure** — do this in a **terminal** (not in the notebook, because `input()` is finicky inside `!python ...`):

```
cd ~/Desktop/robotic-companion
source companion_env/bin/activate
python -m companion.motor.cli recenter
```

1. Torque turns OFF — support the head with your hand (gravity may flop it)
2. Hand-position the head to look forward and level
3. Hold it steady and press ENTER
4. **Power-cycle the servos** after (pull + plug power)

## 7. Verify EEPROM state

After the single-turn + recenter fixes + power-cycle, confirm everything stuck. **Expected values** on both servos:

```
MIN_ANGLE_LIMIT    = 0
MAX_ANGLE_LIMIT    = 4095
MODE               = 0
LOCK               = 1
PRESENT_POSITION   = roughly 2048 (close to center after recenter)
```

If PRESENT_POSITION is way off (e.g. 8, 41478), the recenter either didn't land at center or didn't stick — re-do section 6 with the head held steady at center before pressing ENTER.

In [ ]:
!python -m companion.motor.cli dump-registers

## 8. Run the calibration wizard

GUI wizard, 9 steps. The wizard re-learns the motor→head relationship from scratch and saves to `config.yaml`.

1. **Connect & scan** — should find [1, 2]
2. **Direction test** — torque OFF; jog each motor ±5°. If a jog turns the wrong way, tick the 'invert' box
3. **Rough zero** — torque OFF, hand-hold head at 'looking forward + level', click Capture. *This overrides whatever PRESENT_POSITION reads with your actual preferred center.*
4. **Combined pan/tilt sanity** — torque ON; click presets. If 'pan left' tilts, toggle the invert checkboxes
5. **Limit discovery** — step-jog to each mechanical stop, mark it. Step size 2° is fine
6. **Refine zeros** — pan zero = limit midpoint; set tilt zero with phone level
7. **Gear ratio** — command +500 ticks both, measure actual head pitch, solve
8. **Verify** — sliders + live 3D preview
9. **Save** — writes everything to `config.yaml`

**Success signs**:
- Step 2 direction test: a +5° click moves the pinion a small, visible amount (torque stays off by design in this build — click jogs still send goals)
- Step 4 onward: torque is on, motors hold position. Preset 'Pan LEFT +10°' should move the head a small, visible amount in the expected direction

In [ ]:
!python -m companion.motor.cli calibrate

## 9. Move the head (post-calibration)

Small, verifiable moves. If any of these commands make the head lunge or spin, calibration didn't stick — re-check `config.yaml` values and re-run the wizard.

In [ ]:
!python -m companion.motor.cli test --pan 15 --tilt 0
!python -m companion.motor.cli test --pan 0 --tilt 8
!python -m companion.motor.cli test --pan 0 --tilt 0

## 10. Troubleshooting

### `scan` returns `[]` but adapter is detected

Diagnostic sweep across all common baud rates and all 253 IDs (~3 min):

In [ ]:
!python -m companion.motor.cli diag

### Raw serial ping — is the adapter even transmitting?

Bypasses the SDK and sends a raw ping packet. Useful to distinguish 'adapter dead' vs 'servo silent'.

- Reply `ff ff 01 02 00 fc` (6 bytes) → one servo at ID 1, healthy
- Reply that's the exact packet sent back → half-duplex echo, no servo responding
- Reply of 4 garbage bytes → two servos both at ID 1 colliding (run section 4)
- No bytes at all → adapter or wiring problem

In [ ]:
!python -m companion.motor.cli raw --id 1

### Simulator-only — rehearse without hardware

Slider UI, 3D head preview, no motors required. Good for testing kinematics changes.

In [ ]:
!python -m companion.motor.cli sim